# RQ5 — Model ranking across multiple metrics

**Research question (RQ5).** How do candidate models rank when considering multiple performance metrics (MAE, RMSE, R²) simultaneously?

**Task:** regression to predict `revenue_million`. **Outputs:** CSV ranking table in `./outputs`.

## Methodology (this notebook)

- Fit the same model set as RQ2 on a fixed train/test split.
- Rank by MAE (asc), RMSE (asc), and R² (desc).
- Aggregate ranks (lower is better).

In [1]:
# Setup: paths, load data, modeling frame (Global movies — regression)
from __future__ import annotations

import os
import warnings
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

IS_KAGGLE = os.path.exists("/kaggle/input")
INPUT_ROOT = Path("/kaggle/input") if IS_KAGGLE else Path(".")
OUT = Path("/kaggle/working") if IS_KAGGLE else Path("outputs")
OUT.mkdir(parents=True, exist_ok=True)
RQ_PREFIX = "RQ05"
RANDOM_STATE = 42
TARGET = "revenue_million"

sns.set_theme(style="whitegrid", context="notebook", font_scale=1.0)


def find_raw_table_path() -> Path:
    preferred = ("global_movies_dataset_1950_2026.csv",)
    found: list[Path] = []
    if IS_KAGGLE:
        for root, _, files in os.walk(INPUT_ROOT):
            for fn in files:
                p = Path(root) / fn
                if p.suffix.lower() in {".csv"}:
                    found.append(p)
    else:
        for p in INPUT_ROOT.rglob("*"):
            if p.is_file() and p.suffix.lower() in {".csv"}:
                found.append(p)
    for name in preferred:
        for p in found:
            if p.name.lower() == name.lower():
                return p
    if found:
        return found[0]
    raise FileNotFoundError(
        "No CSV found. Add the dataset via Kaggle Add Input or place global_movies_dataset_1950_2026.csv next to this notebook."
    )


def prepare_modeling_df(raw: pd.DataFrame) -> pd.DataFrame:
    d = raw.copy()
    numeric_cols = [
        "release_year",
        "runtime_min",
        "imdb_rating",
        "votes",
        "budget_million",
        "marketing_budget_million",
        "metascore",
        "audience_score",
        "award_nominations",
        "award_wins",
    ]
    for c in numeric_cols:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")
    if "franchise_flag" in d.columns:
        d["franchise_flag"] = pd.to_numeric(d["franchise_flag"], errors="coerce")

    cat_cols = [
        "genre",
        "subgenre",
        # high-cardinality fields intentionally excluded for speed
        "country",
        "language",
        "streaming_platform",
    ]
    for c in cat_cols:
        if c in d.columns:
            d[c] = d[c].fillna("missing").astype(str)

    d[TARGET] = pd.to_numeric(d[TARGET], errors="coerce")
    d = d.dropna(subset=[TARGET])
    if len(d) > 30000:
        d = d.sample(30000, random_state=RANDOM_STATE)
    return d


def regression_metrics(y_true, y_pred) -> dict:
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(mean_squared_error(y_true, y_pred) ** 0.5),
        "R2": float(r2_score(y_true, y_pred)),
    }


RAW_PATH = find_raw_table_path()
df = pd.read_csv(RAW_PATH)
MODEL_DF = prepare_modeling_df(df)

DEFAULT_FEATURE_COLS = [
    "release_year",
    "runtime_min",
    "imdb_rating",
    "votes",
    "budget_million",
    "marketing_budget_million",
    "metascore",
    "audience_score",
    "award_nominations",
    "award_wins",
    "franchise_flag",
    "genre",
    "subgenre",
    "country",
    "language",
    "streaming_platform",
]
LEAKY_OR_LABEL_COLS = {"roi_pct", "top_100_prob", "blockbuster_flag"}
FEATURE_COLS = [c for c in DEFAULT_FEATURE_COLS if c in MODEL_DF.columns and c not in LEAKY_OR_LABEL_COLS]

print("Loaded:", RAW_PATH)
print("Rows (modeling):", len(MODEL_DF), "Features:", len(FEATURE_COLS))


Loaded: global_movies_dataset_1950_2026.csv
Rows (modeling): 30000 Features: 16


In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVR
from sklearn.tree import DecisionTreeRegressor

X = MODEL_DF[FEATURE_COLS]
y = MODEL_DF[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE
)

from pandas.api.types import is_numeric_dtype

cat_cols = [c for c in FEATURE_COLS if not is_numeric_dtype(MODEL_DF[c])]
num_cols = [c for c in FEATURE_COLS if is_numeric_dtype(MODEL_DF[c])]


def make_preprocessor():
    num_pipe = Pipeline(
        [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
    )
    cat_pipe = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore")),
        ]
    )
    return ColumnTransformer(
        [("num", num_pipe, num_cols), ("cat", cat_pipe, cat_cols)]
    )


models = [
    ("LinearRegression", LinearRegression()),
    ("Ridge", Ridge(random_state=RANDOM_STATE, alpha=2.0)),
    (
        "DecisionTree",
        DecisionTreeRegressor(
            random_state=RANDOM_STATE, max_depth=16, min_samples_leaf=8
        ),
    ),
    ("KNN", KNeighborsRegressor(n_neighbors=9, weights="distance")),
    (
        "RandomForest",
        RandomForestRegressor(
            random_state=RANDOM_STATE,
            n_estimators=250,
            min_samples_leaf=4,
            n_jobs=-1,
        ),
    ),
    (
        "GradientBoosting",
        GradientBoostingRegressor(
            random_state=RANDOM_STATE,
            max_depth=4,
            n_estimators=180,
            learning_rate=0.08,
        ),
    ),
    ("LinearSVR", LinearSVR(max_iter=8000, C=0.6)),
]

try:
    from xgboost import XGBRegressor

    models.insert(
        -1,
        (
            "XGBoost",
            XGBRegressor(
                random_state=RANDOM_STATE,
                n_estimators=500,
                max_depth=7,
                learning_rate=0.05,
                subsample=0.9,
                colsample_bytree=0.9,
                n_jobs=-1,
            ),
        ),
    )
except Exception:
    pass

rows = []
for name, est in models:
    pipe = Pipeline([("prep", make_preprocessor()), ("model", est)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    rows.append({"Model": name, **regression_metrics(y_test, pred)})

tbl = pd.DataFrame(rows)

rank_mae = tbl["MAE"].rank(ascending=True, method="min")
rank_rmse = tbl["RMSE"].rank(ascending=True, method="min")
rank_r2 = tbl["R2"].rank(ascending=False, method="min")

tbl_rank = tbl.copy()
tbl_rank["Rank_MAE"] = rank_mae
tbl_rank["Rank_RMSE"] = rank_rmse
tbl_rank["Rank_R2"] = rank_r2
tbl_rank["AvgRank"] = (rank_mae + rank_rmse + rank_r2) / 3.0
tbl_rank = tbl_rank.sort_values(["AvgRank", "RMSE"])

tbl_rank.to_csv(OUT / f"{RQ_PREFIX}_table_metrics_and_ranks.csv", index=False)

# Figure: average rank by model
fig, ax = plt.subplots(figsize=(8, 4.6))
plot_df = tbl_rank.sort_values("AvgRank")
sns.barplot(data=plot_df, x="AvgRank", y="Model", ax=ax, palette="viridis")
ax.set_title("RQ5 — Average rank across MAE, RMSE, R² (lower is better)")
plt.tight_layout()
fig.savefig(OUT / f"{RQ_PREFIX}_fig_average_rank.pdf")
plt.close()

tbl_rank


,Model,MAE,RMSE,R2,Rank_MAE,Rank_RMSE,Rank_R2,AvgRank
5,GradientBoosting,83.467941,157.262345,0.397498,2.0,1.0,1.0,1.333333
4,RandomForest,84.230010,157.262460,0.397497,3.0,2.0,2.0,2.333333
1,Ridge,89.154621,159.473793,0.380434,5.0,3.0,3.0,3.666667
6,XGBoost,84.602067,159.783031,0.378029,4.0,4.0,4.0,4.000000
7,LinearSVR,78.554171,166.687763,0.323113,1.0,6.0,6.0,4.333333
0,LinearRegression,92.017129,164.167531,0.343426,7.0,5.0,5.0,5.666667
3,KNN,91.058404,168.938203,0.304712,6.0,7.0,7.0,6.666667
2,DecisionTree,94.592246,179.590621,0.214265,8.0,8.0,8.0,8.000000
